# Chess Atlas — Train / Val / Test Split

**Why this notebook exists as a standalone:**  
The split is computed once, saved to CSV, and never re-run during training. Every subsequent script — training, evaluation, EDA — loads the CSV instead of recomputing. This guarantees:
- No accidental data leakage between runs
- Identical splits whether you run locally or on a remote GPU
- A reviewable, diffable artifact you can commit to version control

**Split strategy — Video-level, not frame-level:**  
All 64 squares from any given board come from the same source video. If we split at the square level, the same visual style (piece set, board theme, lighting) leaks across train/val/test — which likely explains the 99% in the previous experiment. Splitting at the **video level** means the model must generalize to unseen visual styles entirely.

```
15 videos  →  Train: 9 videos (~60%)  |  Val: 3 videos (~20%)  |  Test: 3 videos (~20%)
```

With only 15 videos we cannot do fully stratified splits by label (each video has all 13 classes anyway), but we DO stratify by **boards-per-video** to avoid accidentally putting all the data-rich videos in one split.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
# Adjust DATASET_ROOT to wherever your dataset folder lives.
# The metadata CSV is expected at DATASET_ROOT / 'dataset_metadata.csv'
DATASET_ROOT  = Path('.')          # e.g. Path('/data/chess_atlas')
METADATA_PATH = DATASET_ROOT / './dataset/dataset_metadata.csv'
OUTPUT_PATH   = DATASET_ROOT / 'split_assignments.csv'

print(f'Reading metadata from: {METADATA_PATH.resolve()}')

Reading metadata from: /WAVE/users2/unix/jvong/ChessCVModel/dataset/dataset_metadata.csv


/local/scratch/192595/ipykernel_55018/4104654724.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1 · Load & Validate

In [2]:
df = pd.read_csv(METADATA_PATH)

# ── Schema check ───────────────────────────────────────────────────────────────
REQUIRED_COLS = {'video_id', 'board', 'rank', 'file', 'label', 'filepath'}
missing = REQUIRED_COLS - set(df.columns)
assert not missing, f'Metadata is missing columns: {missing}'

# ── Board completeness check (every board must have exactly 64 squares) ────────
board_sizes = df.groupby(['video_id', 'board']).size()
bad_boards  = board_sizes[board_sizes != 64]
if len(bad_boards):
    print(f'WARNING: {len(bad_boards)} boards do not have exactly 64 squares:')
    print(bad_boards)
else:
    print(f'✓  All {len(board_sizes)} boards have exactly 64 squares')

print(f'\nDataset shape : {df.shape}')
print(f'Unique videos : {df["video_id"].nunique()}')
print(f'Unique boards : {len(board_sizes)}')
print(f'Unique labels : {df["label"].nunique()}')

✓  All 110 boards have exactly 64 squares

Dataset shape : (7040, 6)
Unique videos : 15
Unique boards : 110
Unique labels : 13


## 2 · Dataset Profile

In [3]:
print('=== Label Distribution ===')
label_counts = df['label'].value_counts()
label_pct    = (label_counts / len(df) * 100).round(1)
label_summary = pd.DataFrame({'count': label_counts, 'pct': label_pct})
print(label_summary.to_string())

print('\n=== Boards per Video ===')
boards_per_video = df.groupby('video_id')['board'].nunique().sort_values(ascending=False)
print(boards_per_video.to_string())

print(f'\nTotal squares : {len(df):,}')
print(f'Empty squares : {label_counts.get("Empty", 0):,}  ({label_pct.get("Empty", 0):.1f}%)')
print(f'Piece squares : {len(df) - label_counts.get("Empty", 0):,}')

=== Label Distribution ===
             count   pct
label                   
Empty         4461  63.4
WhitePawn      649   9.2
BlackPawn      636   9.0
WhiteRook      186   2.6
BlackRook      174   2.5
BlackBishop    139   2.0
WhiteBishop    137   1.9
WhiteKnight    131   1.9
BlackKnight    130   1.8
WhiteKing      110   1.6
BlackKing      108   1.5
BlackQueen      94   1.3
WhiteQueen      85   1.2

=== Boards per Video ===
video_id
ghJRGPXsjfk    17
PmQs1KhB948    11
-ZVbDR3sRRo    11
r0jMl4qNdyQ    10
65VWIFlc4C4     9
Xxqi7IvwekE     8
CmM1zxS_Ae8     7
u0o38cEaGdw     6
B4lR3NYwI8      6
9dQzTnvsNG4     6
3I_ESQVyxNc     5
NFod-ozimmM     5
Uc7Kf_-hsgw     4
wYCYWpx3CFM     4
QNcO9CJyDBc     1

Total squares : 7,040
Empty squares : 4,461  (63.4%)
Piece squares : 2,579


## 3 · Video-Level Split

Stratify videos into **low / medium / high** board-count tiers, then assign tiers proportionally across splits so no single split gets all the data-rich videos.

In [4]:
# ── Build one row per video with its board count ────────────────────────────────
video_stats = (
    df.groupby('video_id')
      .agg(n_boards=('board', 'nunique'), n_squares=('filepath', 'count'))
      .reset_index()
      .sort_values('n_boards')
)

# ── Tier by board count: low (1-5), mid (6-9), high (10+) ──────────────────────
def tier(n):
    if n <= 5:  return 'low'
    if n <= 9:  return 'mid'
    return 'high'

video_stats['tier'] = video_stats['n_boards'].apply(tier)
print(video_stats[['video_id','n_boards','n_squares','tier']].to_string(index=False))
print(f'\nTier counts: {video_stats["tier"].value_counts().to_dict()}')

   video_id  n_boards  n_squares tier
QNcO9CJyDBc         1         64  low
Uc7Kf_-hsgw         4        256  low
wYCYWpx3CFM         4        256  low
3I_ESQVyxNc         5        320  low
NFod-ozimmM         5        320  low
u0o38cEaGdw         6        384  mid
 B4lR3NYwI8         6        384  mid
9dQzTnvsNG4         6        384  mid
CmM1zxS_Ae8         7        448  mid
Xxqi7IvwekE         8        512  mid
65VWIFlc4C4         9        576  mid
r0jMl4qNdyQ        10        640 high
PmQs1KhB948        11        704 high
-ZVbDR3sRRo        11        704 high
ghJRGPXsjfk        17       1088 high

Tier counts: {'mid': 6, 'low': 5, 'high': 4}


In [5]:
# ── Assign split within each tier ──────────────────────────────────────────────
# Target: ~60% train / ~20% val / ~20% test at the VIDEO level.
# With 15 videos: 9 train, 3 val, 3 test is the natural fit.
# We shuffle within each tier then slice proportionally.

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.20  # test gets the remainder

split_map = {}  # video_id -> 'train' | 'val' | 'test'

for tier_name, group in video_stats.groupby('tier'):
    vids = group['video_id'].tolist()
    # Shuffle within tier using the seeded rng
    order = rng.permutation(len(vids))
    vids  = [vids[i] for i in order]

    n       = len(vids)
    n_train = max(1, round(n * TRAIN_FRAC))
    n_val   = max(1, round(n * VAL_FRAC))
    # Ensure we don't exceed available videos
    n_val   = min(n_val,  n - n_train)
    n_test  = n - n_train - n_val

    for vid in vids[:n_train]:             split_map[vid] = 'train'
    for vid in vids[n_train:n_train+n_val]:split_map[vid] = 'val'
    for vid in vids[n_train+n_val:]:       split_map[vid] = 'test'

video_stats['split'] = video_stats['video_id'].map(split_map)

print('=== Video → Split Assignment ===')
print(video_stats[['video_id','tier','n_boards','split']].sort_values('split').to_string(index=False))

=== Video → Split Assignment ===
   video_id tier  n_boards split
Uc7Kf_-hsgw  low         4  test
CmM1zxS_Ae8  mid         7  test
r0jMl4qNdyQ high        10  test
wYCYWpx3CFM  low         4 train
3I_ESQVyxNc  low         5 train
NFod-ozimmM  low         5 train
u0o38cEaGdw  mid         6 train
 B4lR3NYwI8  mid         6 train
9dQzTnvsNG4  mid         6 train
65VWIFlc4C4  mid         9 train
-ZVbDR3sRRo high        11 train
ghJRGPXsjfk high        17 train
QNcO9CJyDBc  low         1   val
Xxqi7IvwekE  mid         8   val
PmQs1KhB948 high        11   val


## 4 · Validate the Split

In [6]:
# ── Attach split to every square ───────────────────────────────────────────────
df['split'] = df['video_id'].map(split_map)

assert df['split'].isna().sum() == 0, 'Some rows have no split assignment!'

print('=== Squares per Split ===')
split_counts = df['split'].value_counts()
split_pct    = (split_counts / len(df) * 100).round(1)
print(pd.DataFrame({'squares': split_counts, 'pct': split_pct}).to_string())

print('\n=== Videos per Split ===')
print(df.groupby('split')['video_id'].nunique())

print('\n=== Boards per Split ===')
print(df.groupby('split').apply(lambda x: x.groupby(['video_id','board']).ngroups))

=== Squares per Split ===
       squares   pct
split               
train     4416  62.7
test      1344  19.1
val       1280  18.2

=== Videos per Split ===
split
test     3
train    9
val      3
Name: video_id, dtype: int64

=== Boards per Split ===
split
test     21
train    69
val      20
dtype: int64


/local/scratch/192595/ipykernel_55018/522174563.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby('split').apply(lambda x: x.groupby(['video_id','board']).ngroups))


In [7]:
# ── Label distribution per split (catch class-starvation in val/test) ──────────
print('=== Label Distribution by Split ===')
label_split = (
    df.groupby(['split', 'label'])
      .size()
      .unstack(fill_value=0)
      .T                         # labels as rows for readability
)
# Reorder splits
label_split = label_split[['train', 'val', 'test']]
print(label_split.to_string())

# Warn if any class has zero examples in val or test
for split_name in ['val', 'test']:
    zero_classes = label_split[label_split[split_name] == 0].index.tolist()
    if zero_classes:
        print(f'\n⚠  WARNING: {split_name} is missing these classes: {zero_classes}')
        print('   Consider re-running with a different SEED or adjusting video assignment.')
    else:
        print(f'\n✓  All 13 classes present in {split_name}')

=== Label Distribution by Split ===
split        train  val  test
label                        
BlackBishop     95   23    21
BlackKing       69   20    19
BlackKnight     87   16    27
BlackPawn      417  105   114
BlackQueen      55   18    21
BlackRook      117   24    33
Empty         2732  859   870
WhiteBishop     93   18    26
WhiteKing       69   20    21
WhiteKnight     83   24    24
WhitePawn      426  107   116
WhiteQueen      57   11    17
WhiteRook      116   35    35

✓  All 13 classes present in val

✓  All 13 classes present in test


In [8]:
# ── Leakage check: no video_id should appear in more than one split ────────────
video_split_counts = df.groupby('video_id')['split'].nunique()
leaked = video_split_counts[video_split_counts > 1]
if len(leaked):
    print(f'CRITICAL LEAKAGE DETECTED in: {leaked.index.tolist()}')
else:
    print('✓  No video appears in more than one split — zero leakage')

✓  No video appears in more than one split — zero leakage


## 5 · Save Split CSV

In [9]:
# ── Save the full metadata with split column attached ──────────────────────────
df.to_csv(OUTPUT_PATH, index=False)
print(f'✓  Saved {len(df):,} rows → {OUTPUT_PATH.resolve()}')
print()
print('Columns:', df.columns.tolist())
print()
print('Preview:')
display(df.head(5))

✓  Saved 7,040 rows → /WAVE/users2/unix/jvong/ChessCVModel/split_assignments.csv

Columns: ['video_id', 'board', 'rank', 'file', 'label', 'filepath', 'split']

Preview:


,video_id,board,rank,file,label,filepath,split
0,-ZVbDR3sRRo,0,0,0,Empty,-ZVbDR3sRRo/-ZVbDR3sRRo_Board0_Pos0-0_PieceEmp...,train
1,-ZVbDR3sRRo,0,0,1,Empty,-ZVbDR3sRRo/-ZVbDR3sRRo_Board0_Pos0-1_PieceEmp...,train
2,-ZVbDR3sRRo,0,0,2,Empty,-ZVbDR3sRRo/-ZVbDR3sRRo_Board0_Pos0-2_PieceEmp...,train
3,-ZVbDR3sRRo,0,0,3,Empty,-ZVbDR3sRRo/-ZVbDR3sRRo_Board0_Pos0-3_PieceEmp...,train
4,-ZVbDR3sRRo,0,0,4,BlackRook,-ZVbDR3sRRo/-ZVbDR3sRRo_Board0_Pos0-4_PieceBla...,train


## 6 · How to Load This in Training

```python
import pandas as pd

split_df = pd.read_csv('split_assignments.csv')

train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)

# test_df should NEVER be touched until final evaluation.
# All model selection and hyperparameter tuning uses train/val only.
```

### Key rules to follow downstream
1. **Augmentation applies only to `train_df`** — never to val or test.
2. **Normalization stats (mean/std) are computed on `train_df` only**, then applied to val and test.
3. **Do not re-run this notebook** during an experiment. Load `split_assignments.csv` instead.
4. If you collect new videos, append them to metadata and re-run *this notebook only*, with the same `SEED`.
5. `test_df` is touched **once** — at the very end, when reporting final numbers.

In [10]:
# ── Reproducibility fingerprint ────────────────────────────────────────────────
# Store a hash of the split assignments so you can verify consistency later.
import hashlib, json

fingerprint_data = df[['filepath','split']].sort_values('filepath').to_json(orient='records')
split_hash = hashlib.sha256(fingerprint_data.encode()).hexdigest()[:12]

meta = {
    'seed':       SEED,
    'n_videos':   int(df['video_id'].nunique()),
    'n_boards':   int(df.groupby(['video_id','board']).ngroups),
    'n_squares':  int(len(df)),
    'split_hash': split_hash,
    'split_counts': df['split'].value_counts().to_dict(),
}

meta_path = DATASET_ROOT / 'split_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('=== Split Fingerprint ===')
print(json.dumps(meta, indent=2))
print(f'\n✓  Saved fingerprint → {meta_path.resolve()}')
print()
print('To verify a future run matches this split:')
print(f'  Expected hash: {split_hash}')

=== Split Fingerprint ===
{
  "seed": 42,
  "n_videos": 15,
  "n_boards": 110,
  "n_squares": 7040,
  "split_hash": "527c83f2972c",
  "split_counts": {
    "train": 4416,
    "test": 1344,
    "val": 1280
  }
}

✓  Saved fingerprint → /WAVE/users2/unix/jvong/ChessCVModel/split_meta.json

To verify a future run matches this split:
  Expected hash: 527c83f2972c
